In [1]:
import kagglehub
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer




# =========================
# 2. Wczytanie CSV
# =========================

csv_path = "games.csv"

correct_columns = [
    'AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU',
    'Required age', 'Price', 'Discount', 'DLC count', 'About the game',
    'Supported languages', 'Full audio languages', 'Reviews', 'Header image',
    'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux',
    'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative',
    'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever',
    'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks',
    'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies'
]

# Wczytujemy plik, ignorując zepsuty nagłówek (header=0) i używając naszych kolumn
# #z powodow ograniczen sprzetowych liczba wierszów okrojona
# df = pd.read_csv(csv_path, nrows=100000, names=correct_columns, header=0, index_col=False)
kagglehub.dataset_download('fronkongames/steam-games-dataset', path='games.csv', output_dir='../data')

df = pd.read_csv("../data/games.csv", encoding='latin-1', nrows=100000, names=correct_columns, header=0, index_col=False)
target = "Name"

# Usuwamy rekordy bez nazwy gry
df = df.dropna(subset=[target])

# Zamieniamy puste wartości na pusty tekst
df = df.fillna("")

print("Liczba wczytanych gier:", len(df))
print("Kolumny:", df.columns.tolist())
df.head(5)
print(df['Peak CCU'].dtype)
#usuwanie gier, ktore maja bardzo male zainteresowanie
df = df[df['Peak CCU'] >= 100]

Liczba wczytanych gier: 99999
Kolumny: ['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'Discount', 'DLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']
int64


In [2]:
sorted_df = df.sort_values(by=["Positive"], ascending=False)
sorted_df.head(5)

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
45509,730,Counter-Strike 2,"Aug 21, 2012",100000000 - 200000000,1013936,0,0.00,0,1,"For over two decades, Counter-Strike has offer...",...,702,6237,307,Valve,Valve,"Multi-player,Cross-Platform Multiplayer,Steam ...","Action,Free To Play","FPS,Shooter,Multiplayer,Competitive,Action,Tea...",https://shared.akamai.steamstatic.com/store_it...,
4193,570,Dota 2,"Jul 9, 2013",100000000 - 200000000,623941,0,0.00,0,2,"The most-played game on Steam. Every day, mill...",...,1427,1155,860,Valve,Valve,"Multi-player,Co-op,Steam Trading Cards,Steam W...","Action,Strategy,Free To Play","Free to Play,MOBA,Multiplayer,Strategy,e-sport...",https://shared.akamai.steamstatic.com/store_it...,
59122,271590,Grand Theft Auto V Legacy,"Apr 13, 2015",50000000 - 100000000,67851,17,0.00,0,0,"When a young street hustler, a retired bank ro...",...,872,5846,158,Rockstar North,Rockstar Games,"Single-player,Multi-player,PvP,Online PvP,Co-o...","Action,Adventure","Open World,Action,Multiplayer,Crime,Automobile...",https://shared.akamai.steamstatic.com/store_it...,
8878,578080,PUBG: BATTLEGROUNDS,"Dec 21, 2017",100000000 - 200000000,314682,13,0.00,0,0,LAND Â Â Drop into an ever-growing and chang...,...,730,6082,302,PUBG Corporation,"KRAFTON, Inc.","Multi-player,PvP,Online PvP,Stats,Remote Play ...","Action,Adventure,Massively Multiplayer,Free To...","Survival,Shooter,Battle Royale,Multiplayer,FPS...",https://shared.akamai.steamstatic.com/store_it...,
10927,105600,Terraria,"May 16, 2011",20000000 - 50000000,24580,0,4.99,50,2,"Dig, Fight, Explore, Build: The very world is ...",...,622,2051,180,Re-Logic,Re-Logic,"Single-player,Multi-player,PvP,Online PvP,Co-o...","Action,Adventure,Indie,RPG","Open World Survival Craft,Sandbox,Survival,2D,...",https://shared.akamai.steamstatic.com/store_it...,


In [3]:
import os

# =========================
# 3. Przygotowanie danych do wyszukiwania
# =========================

def row_to_text(row):
    # ręczne okrajanie dla vektora
    parts = [
        f"Title: {row['Name']}",
        f"Tags: {row['Tags']}",      # Tagi na 2. miejscu! Model od razu je przeczyta.
        f"Genres: {row['Genres']}",  # Gatunki na 3. miejscu!
        f"Categories: {row['Categories']}",
        # Opis gry z obcieciem do 500
        f"About: {str(row['About the game'])[:500]}"
    ]

    return " | ".join(parts)
df["game_description"] = df.apply(row_to_text, axis=1)


# 1. Ładujemy mały, bardzo szybki model do zamiany tekstu na wektory
# 'all-MiniLM-L6-v2' tworzy wektory o wielkości 384 wymiarów
print("Ładowanie modelu embeddingowego...")
# wielojezyczny model
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cuda')
index_file = "games_faiss.index"

# 2. Sprawdzamy, czy plik już istnieje na dysku
if os.path.exists(index_file):
    print(f"Znaleziono zapisany indeks! Wczytywanie z {index_file}...")

    # Wczytanie z dysku (trwa ułamek sekundy!)
    index = faiss.read_index(index_file)
    print(f"Wczytano {index.ntotal} gier błyskawicznie!")

else:
    print("Brak zapisanego indeksu. Zamiana opisów na wektory (to potrwa chwilę)...")

    # Przeliczanie wektorów jesli nie bylo pliku
    embeddings = embedder.encode(df["game_description"].tolist(), show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')

    print("Budowa bazy wektorowej FAISS...")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    # Magia: Zapisujemy gotowy indeks na dysk!
    print(f"Zapisywanie wektorów do pliku {index_file}...")
    faiss.write_index(index, index_file)
    print("Gotowe! Przy następnym uruchomieniu skrypt pominie ten krok.")

Ładowanie modelu embeddingowego...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Znaleziono zapisany indeks! Wczytywanie z games_faiss.index...
Wczytano 1631 gier błyskawicznie!


In [5]:
#Zaczytywanie Qwen z wyuczonym plastrem
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from peft import PeftModel

print("Czy GPU jest dostępne?", torch.cuda.is_available())

# ==========================================
# 1. Ścieżki
# ==========================================
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "./qwen_gotowy" #miejsce wyuczonego plastra

# ==========================================
# 2. Kompresja 4-bit (Zabezpieczenie 6GB VRAM)
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# ==========================================
# 3. Składanie modelu (Baza + Plaster)
# ==========================================
print("Ładowanie bazy...")
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# Ładujemy czystą bazę
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print("Naklejanie plastra LoRA...")
# doklejanie wytrenowanego plastra
model = PeftModel.from_pretrained(base_model, adapter_path)

# ==========================================
# 4. Tworzenie Pipeline'u
# ==========================================
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
    #max_new_tokens=256, # Ile maksymalnie słów ma wygenerować
    #temperature=0.7     # Stopień "kreatywności" (0.1 = robot, 1.0 = fantasta)
)


Czy GPU jest dostępne? True
Ładowanie bazy...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Naklejanie plastra LoRA...


In [6]:
# =========================
# 4. Funkcja wyszukująca gry
# =========================

def find_best_games_faiss(user_question, top_k=5):
    # 1. FAISS wyciąga z bazy więcej gier (tzw. over-fetching)
    fetch_k = 500
    question_vector = embedder.encode([user_question]).astype('float32')
    distances, indices = index.search(question_vector, fetch_k)

    best_indices = indices[0]
    best_distances = distances[0]

    # 2. Składamy te 500 wyników do tabeli
    results = df.iloc[best_indices].copy()
    results["distance"] = best_distances

    # 3. Upewniamy się, że kolumna 'Positive' to na pewno liczby (a nie tekst)
    results['Positive'] = pd.to_numeric(results['Positive'], errors='coerce').fillna(0)

    #Reranking na podstawie wyniku wektoryzacji(odleglosci) oraz liczby recenzji
    #Zeby uniknac sytuacji w ktorej gra ma swietne opisy tagi, ale sama gra jest slaba

    # A. Normalizujemy odległość FAISS (im mniejsza, tym bliżej 1.0)
    max_dist = results['distance'].max()
    min_dist = results['distance'].min()
    if max_dist == min_dist:
        results['norm_dist'] = 1.0
    else:
        results['norm_dist'] = 1 - ((results['distance'] - min_dist) / (max_dist - min_dist))

    # B. Normalizujemy recenzje 'Positive' (im więcej, tym bliżej 1.0)
    max_pos = results['Positive'].max()
    min_pos = results['Positive'].min()
    if max_pos == min_pos:
        results['norm_pos'] = 0.0
    else:
        results['norm_pos'] = (results['Positive'] - min_pos) / (max_pos - min_pos)

    # C. Obliczamy Ostateczny Wynik (FINAL SCORE)
    # Ustalanie wag

    results['final_score'] = (results['norm_dist'] * 0.5) + (results['norm_pos'] * 0.5)

    # ==========================================

    # 4. Sortujemy listę malejąco według Ostatecznego Wyniku i ucinamy do naszej 'top_k' (czyli 5)
    results = results.sort_values(by='final_score', ascending=False).head(top_k)


    return results

In [7]:
# =========================
# 5. Funkcja odpowiedzi LLM
# =========================

def recommend_game(user_question, top_k=5):
    best_games = find_best_games_faiss(user_question, top_k=top_k)

    context = ""

    for i, row in best_games.iterrows():
        context += f"Tytuł: {row[target]}\n"
        # Podajemy mu TYLKO kluczowe cechy zamiast 40 kolumn
        context += f"Gatunek: {row['Genres']} | Tagi: {row['Tags']}\n"
        # Ucinamy opis gry do maksymalnie 200 znaków
        context += f"Opis: {str(row['About the game'])[:200]}...\n"
        context += f"Dopasowanie: {row['distance']:.4f}\n"
        context += f"opinie: {row['Positive']:.4f}\n\n"

    prompt = f"""
            Jesteś systemem rekomendacji gier.

            Użytkownik pyta:
            {user_question}

            Na podstawie poniższych danych z pliku CSV wybierz najbardziej odpowiedni tytuł gry.
            Nie wymyślaj gry spoza danych.
            Odpowiedz po polsku.
            Podaj:
            1. najlepszy tytuł gry,
            2. krótkie uzasadnienie,
            3. ewentualnie 2 alternatywy.

            Dane z CSV:
            {context}s
            """

    messages = [
        {
            "role": "system",
            "content": "Jesteś pomocnym asystentem do rekomendowania gier na podstawie danych CSV."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    output = pipe(
        messages,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    return output[0]["generated_text"][-1]["content"], best_games[[target, "distance"]]

df.head(5)

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies,game_description
172,1272080,PAYDAY 3,"Sep 21, 2023",2000000 - 5000000,644,17,14.99,50,22,Step out of retirement back into the life of c...,...,721,194,Starbreeze Studios,Starbreeze Entertainment,"Single-player,Multi-player,Co-op,Online Co-op,...","Action,Adventure,RPG","Heist,Co-op,Crime,Action,Multiplayer,FPS,Shoot...",https://shared.akamai.steamstatic.com/store_it...,,"Title: PAYDAY 3 | Tags: Heist,Co-op,Crime,Acti..."
261,1517290,Battlefieldâ¢ 2042,"Nov 19, 2021",10000000 - 20000000,2774,17,59.99,0,2,WELCOME TO 2042 Battlefieldâ¢ 2042 is a first...,...,1134,116,DICE,Electronic Arts,"Multi-player,PvP,Online PvP,Co-op,Online Co-op...","Action,Adventure,Casual","FPS,Multiplayer,Shooter,Military,War,Singlepla...",https://shared.akamai.steamstatic.com/store_it...,,"Title: Battlefieldâ¢ 2042 | Tags: FPS,Multipl..."
332,3070880,Farming & Supermarket Simulator,"Aug 31, 2025",0 - 20000,146,0,8.39,0,0,'Farming &amp; Supermarket Simulator' is a del...,...,1119,0,Bull Games,Bull Games,"Single-player,Steam Achievements,Family Sharing","Adventure,Casual,Indie,Simulation","Early Access,Relaxing,Simulation,Economy,Manag...",https://shared.akamai.steamstatic.com/store_it...,,Title: Farming & Supermarket Simulator | Tags:...
352,1147860,UFO 50,"Sep 18, 2024",100000 - 200000,163,0,18.74,25,1,UFO 50 is a collection of 50 single and multip...,...,230,9,Mossmouth,Mossmouth,"Single-player,Multi-player,PvP,Shared/Split Sc...","Action,Adventure,Indie,RPG,Strategy","Indie,Pixel Graphics,Action,Retro,Strategy,RPG...",https://shared.akamai.steamstatic.com/store_it...,,"Title: UFO 50 | Tags: Indie,Pixel Graphics,Act..."
582,339600,VEGA Conflict,"Dec 14, 2015",1000000 - 2000000,290,0,0.00,0,0,"Stake your claim, command your fleets, and wag...",...,231,0,KIXEYE,KIXEYE,"Multi-player,MMO,PvP,Online PvP,Cross-Platform...","Massively Multiplayer,Strategy,Free To Play","Free to Play,2D,Space,Massively Multiplayer,St...",https://shared.akamai.steamstatic.com/store_it...,,"Title: VEGA Conflict | Tags: Free to Play,2D,S..."


In [13]:
# =========================
# 6. Pobieranie pytania z konsoli
# =========================

while True:
    question = input("\nNapisz, jakiej gry szukasz albo wpisz 'exit': ")

    if question.lower() in ["exit", "quit", "koniec"]:
        break

    answer, matched_games = recommend_game(question, top_k=5)

    print("\nNajbardziej podobne gry z CSV:")
    print(matched_games)

    print("\nOdpowiedź modelu:")
    print(answer)

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Najbardziej podobne gry z CSV:
                                           Name   distance
45509                          Counter-Strike 2  17.030691
60313                RaceRoom Racing Experience   9.718015
25963                 NIGHT-RUNNERSâ¢ PROLOGUE  11.203672
36070                Assetto Corsa Competizione  11.303827
6629   Need for Speedâ¢ Hot Pursuit Remastered  11.664002

Odpowiedź modelu:
Na podstawie otrzymanych danych dla gracza wygląda na to, że chciałby wybrać się w wyścigu.
Zdecydowana jest graczka **Need for Speed Hot Pursuit Remastered**.
Jednak też mogą być interesujace **Assetto Corsa Competizione** i **Night Runners Prologue**, które są również bardzo atrakcyjne.
Niech będzie **Counter-Strike 2** jako dodatkowa opcja, ale nie jest to główny wybór.
Odpowiedź: **Need for Speed Hot Pursuit Remastered**
